In [4]:
import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [5]:
df = pd.read_csv("../../data/processed/final_dataset.csv")

In [8]:
print(df.columns)

Index(['summary', 'description', 'priority.name', 'status.name',
       'resolution.name', 'issuetype.name'],
      dtype='object')


In [9]:
print(df["summary"].isnull().sum())
print(df["description"].isnull().sum())

18100
123243


In [10]:
df["summary"] = df["summary"].fillna("")
df["description"] = df["description"].fillna("")

In [11]:
df["text"] = df["summary"] + " " + df["description"]

In [12]:
print(df["text"].isnull().sum())

0


In [13]:
df = df.dropna(subset=["text"])

In [14]:
print(df["text"].isnull().sum())

0


In [16]:
df.rename(columns={"priority.name": "target"}, inplace=True)

In [17]:
print(df.columns.tolist())

['summary', 'description', 'target', 'status.name', 'resolution.name', 'issuetype.name', 'text']


In [18]:
X = df["text"]
y = df["target"]

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2
)

X_features = tfidf.fit_transform(X)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
) 

In [21]:
print(X_train.shape)
print(X_test.shape)

(919458, 3000)
(229865, 3000)


In [22]:
print(df.shape)

(1149323, 7)


In [23]:
rf_model = RandomForestClassifier(
    n_estimators=10,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [24]:
rf_pred = rf_model.predict(X_test)

In [25]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, rf_pred)
print("Accuracy:", accuracy)

Accuracy: 0.6541535248950471


In [26]:
with open("../../models/random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

In [27]:
import xgboost

print(xgboost.__version__)

3.2.0


In [28]:
import multiprocessing
print(multiprocessing.cpu_count())

12


In [30]:
%whos

Variable                 Type                      Data/Info
------------------------------------------------------------
RandomForestClassifier   ABCMeta                   <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
TfidfVectorizer          type                      <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
X                        Series                    0          Update config <...>h: 1149323, dtype: object
XGBClassifier            type                      <class 'xgboost.sklearn.XGBClassifier'>
X_features               csr_matrix                <Compressed Sparse Row sp<...>2126)	0.09122372795083646
X_test                   csr_matrix                <Compressed Sparse Row sp<...> 1196)	0.5568481810980134
X_train                  csr_matrix                <Compressed Sparse Row sp<...> 581)	0.26154102040014193
accuracy                 float                     0.6541535248950471
accuracy_score           function                  <function accuracy_score

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

In [50]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["target"] = label_encoder.fit_transform(df["target"])

In [51]:
X = df["text"]
y = df["target"]

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [53]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)
print(y_train.head())
print(y_train.dtype)
xgb_model.fit(X_train_tfidf, y_train)


383375    4
213743    4
229736    4
262125    4
203878    4
Name: target, dtype: int64
int64


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [54]:
print(type(X_train))
print(type(X_train_tfidf))
print(type(y_train))

print(y_train.head())

<class 'pandas.core.series.Series'>
<class 'scipy.sparse._csr.csr_matrix'>
<class 'pandas.core.series.Series'>
383375    4
213743    4
229736    4
262125    4
203878    4
Name: target, dtype: int64


In [55]:
print(y_train.head())

383375    4
213743    4
229736    4
262125    4
203878    4
Name: target, dtype: int64


In [56]:
print(type(X_train))

<class 'pandas.core.series.Series'>


In [57]:
print(type(X_train_tfidf))

<class 'scipy.sparse._csr.csr_matrix'>


In [58]:
print(df["target"].head())

0    5
1    0
2    1
3    4
4    4
Name: target, dtype: int64


In [59]:
hasattr(xgb_model, "_Booster")

True

In [63]:
xgb_pred = xgb_model.predict(X_test_tfidf)

In [64]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred, average="weighted", zero_division=0))
print("Recall   :", recall_score(y_test, xgb_pred, average="weighted", zero_division=0))
print("F1 Score :", f1_score(y_test, xgb_pred, average="weighted", zero_division=0))

print("\nClassification Report:\n")
print(classification_report(y_test, xgb_pred, zero_division=0))

Accuracy : 0.6651121310334327
Precision: 0.6296877543841265
Recall   : 0.6651121310334327
F1 Score : 0.5488733376655168

Classification Report:

              precision    recall  f1-score   support

           0       0.55      0.01      0.02      7295
           1       0.52      0.02      0.03      9903
           2       0.00      0.00      0.00        35
           3       0.47      0.10      0.17      1366
           4       0.67      0.99      0.80    149258
           5       0.56      0.03      0.06     43324
           6       0.65      0.56      0.61      2711
           7       0.14      0.00      0.00       473
           8       0.00      0.00      0.00        55
           9       0.57      0.23      0.33       168
          10       0.46      0.29      0.35      1356
          11       0.52      0.34      0.41      1274
          12       0.50      0.02      0.03        60
          13       0.50      0.00      0.00      6080
          14       0.72      0.10      0.18 

In [65]:
import pickle

with open("../../models/xgboost.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

print("XGBoost model saved successfully!")

XGBoost model saved successfully!


In [66]:
import pickle

# Save TF-IDF Vectorizer
with open("../../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save Random Forest Model
with open("../../models/random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

# Save XGBoost Model
with open("../../models/xgboost.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

print("TF-IDF Vectorizer Saved")
print("Random Forest Model Saved")
print("XGBoost Model Saved")

TF-IDF Vectorizer Saved
Random Forest Model Saved
XGBoost Model Saved


In [2]:
import os
import joblib

desktop_path = os.path.expanduser("~/Desktop/Rescued_Models")
os.makedirs(desktop_path, exist_ok=True)

# Replace 'rf' and 'vectorizer' with YOUR exact notebook variables:
joblib.dump(rf, os.path.join(desktop_path, "random_forest.pkl"))
joblib.dump(vectorizer, os.path.join(desktop_path, "tfidf_vectorizer.pkl"))

print("🎉 Success! Check your Desktop for 'Rescued_Models'!")

NameError: name 'rf' is not defined